In [1]:
%reload_ext autoreload
%autoreload 2

In [2]:
import datetime
import json
import os
import random

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import wandb
from accelerate.commands.config.update import description
from transformers import (
    AutoModelForMaskedLM,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

def set_seed(seed=42) -> None:
    """Set all seeds to make results reproducible (deterministic mode).
    When seed is a false-y value or not supplied, disables deterministic mode."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

from dotenv import load_dotenv
load_dotenv()

True

In [3]:
for i in range(torch.cuda.device_count()):
    print(torch.cuda.get_device_properties(i).name)

# os.environ["MY_VARIABLE"] = "my_value"


NVIDIA RTX A5000
NVIDIA RTX A5000
NVIDIA RTX A5000
NVIDIA RTX A5000


## Load trained model from wandb

In [5]:
def load_model_from_wandb(model_path):
    run = wandb.init()
    artifact = run.use_artifact(model_path, type='model')
    artifact_dir = artifact.download()
    model = AutoModelForMaskedLM.from_pretrained(artifact_dir)
    return model, artifact_dir

def load_model_from_local(model_path):
    model = AutoModelForMaskedLM.from_pretrained(model_path)
    return model, model_path

def load_model(model_path, use_wandb=True):
    if use_wandb:
        return load_model_from_wandb(model_path)
    else:
        return load_model_from_local(model_path)
    
# checking if two hugging face models are same or not
model_from_wandb, model_from_wandb_path = load_model_from_wandb("nasa-impact/mlm-fine-tuning/model-zymz8ir5:v1")
model_from_local, model_from_local_path = load_model_from_local("/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/tmp/models/timestamp_20250228_18-02-05/answerdotai/ModernBERT-base/checkpoint-296000")


# Compare the state_dicts
# are_models_identical = all(
#     torch.equal(param1, param2) 
#     for param1, param2 in zip(model_from_wandb.state_dict().values(), model_from_local.state_dict().values())
# )


state_dict_1 = model_from_wandb.state_dict()
state_dict_2 = model_from_local.state_dict()


same_weights = all(torch.equal(state_dict_1[k], state_dict_2[k]) for k in state_dict_1.keys())
same_weights

# print(f"Are the models identical? {are_models_identical}")



wandb: Downloading large artifact model-zymz8ir5:v1, 570.92MB. 4 files... 
wandb:   4 of 4 files downloaded.  
Done. 0:0:1.4


True

In [ ]:
import torch
from transformers import AutoModel

model_1 = AutoModel.from_pretrained(nasa-impact/indus-sde-v0.1)
model_2 = AutoModel.from_pretrained(model_name_2)

state_dict_1 = model_1.state_dict()
state_dict_2 = model_2.state_dict()

same_weights = all(torch.equal(state_dict_1[k], state_dict_2[k]) for k in state_dict_1.keys())

print(same_weights)  # True if weights are identical


wandb: WARNING Calling wandb.login() after wandb.init() has no effect.
wandb: Downloading large artifact model-zymz8ir5:v1, 570.92MB. 4 files... 
wandb:   4 of 4 files downloaded.  
Done. 0:0:0.9


ValueError: Artifact name may only contain alphanumeric characters, dashes, underscores, and dots. Invalid name: nasa-impact/mlm-fine-tuning/model-zymz8ir5

In [8]:
from transformers import AutoModelForMaskedLM
import torch

def compare_huggingface_models(model_name1, model_name2):
    # Load both models
    model1 = AutoModelForMaskedLM.from_pretrained(model_name1)
    model2 = AutoModelForMaskedLM.from_pretrained(model_name2)
    
    # Compare architectures
    if model1.config != model2.config:
        return False, "Models have different architectures"
    
    # Get state dictionaries
    state_dict1 = model1.state_dict()
    state_dict2 = model2.state_dict()
    
    # Compare parameter names
    if set(state_dict1.keys()) != set(state_dict2.keys()):
        return False, "Models have different parameter names"
    
    # Compare parameter values
    differences = []
    for param_name in state_dict1.keys():
        tensor1 = state_dict1[param_name]
        tensor2 = state_dict2[param_name]
        
        # Calculate mean squared error between tensors
        difference = torch.nn.functional.mse_loss(tensor1, tensor2)
        if difference.item() > 1e-10:  # Using small epsilon for floating point comparison
            differences.append((param_name, difference.item()))
    
    return len(differences) == 0, differences if differences else "Models are identical"

# Example usage
model1_name = "nasa-impact/indus-sde-v0.1"
model2_name = "/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/general_analysis/artifacts/model-w68aew2o:v1"

are_equal, result = compare_huggingface_models(model1_name, model2_name)
print(f"Are models equal? {are_equal}")
if not are_equal:
    print("\nDifferences found:")
    for param_name, diff in result:
        print(f"{param_name}: {diff:.2e}")

Are models equal? False

Differences found:


ValueError: not enough values to unpack (expected 2, got 1)

In [12]:
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
from collections import OrderedDict

def load_model_and_tokenizer(model_name):
    """Load a model and tokenizer from Hugging Face."""
    try:
        tokenizer = AutoTokenizer.from_pretrained("nasa-impact/indus-sde-v0.1")
        model = AutoModel.from_pretrained(model_name)
        return model, tokenizer
    except Exception as e:
        print(f"Error loading {model_name}: {e}")
        return None, None

def compare_model_architectures(model1, model2):
    """Compare the architectures of two models."""
    # Check if models have the same number of parameters
    params1 = sum(p.numel() for p in model1.parameters())
    params2 = sum(p.numel() for p in model2.parameters())
    
    if params1 != params2:
        print(f"Models have different parameter counts: {params1} vs {params2}")
        return False
    
    # Compare model structure (layers, dimensions, etc.)
    equal_architecture = True
    state_dict1 = model1.state_dict()
    state_dict2 = model2.state_dict()
    
    # Check if the models have the same layers
    if set(state_dict1.keys()) != set(state_dict2.keys()):
        print("Models have different layer structures")
        different_keys = set(state_dict1.keys()).symmetric_difference(set(state_dict2.keys()))
        print(f"Different layers: {different_keys}")
        equal_architecture = False
    
    # Check if shapes of corresponding layers match
    for key in state_dict1:
        if key in state_dict2:
            if state_dict1[key].shape != state_dict2[key].shape:
                print(f"Layer {key} has different shapes: {state_dict1[key].shape} vs {state_dict2[key].shape}")
                equal_architecture = False
    
    return equal_architecture

def compare_model_weights(model1, model2, tolerance=1e-5):
    """Compare the weights of two models."""
    state_dict1 = model1.state_dict()
    state_dict2 = model2.state_dict()
    
    weight_differences = []
    for key in state_dict1:
        if key in state_dict2:
            if state_dict1[key].shape == state_dict2[key].shape:
                # Calculate absolute differences between weights
                diff = torch.abs(state_dict1[key] - state_dict2[key])
                max_diff = torch.max(diff).item()
                mean_diff = torch.mean(diff).item()
                weight_differences.append((key, max_diff, mean_diff))
    
    # Sort by maximum difference
    weight_differences.sort(key=lambda x: x[1], reverse=True)
    
    # Check if all weights are within tolerance
    all_within_tolerance = all(diff[1] < tolerance for diff in weight_differences)
    
    # Print top differences
    print(f"Top 5 layer differences (max absolute difference):")
    for i, (layer, max_diff, mean_diff) in enumerate(weight_differences[:5]):
        print(f"{layer}: max_diff={max_diff:.8f}, mean_diff={mean_diff:.8f}")
    
    return all_within_tolerance, weight_differences

def compare_model_outputs(model1, model2, tokenizer1, tokenizer2, test_sentences):
    """Compare the outputs of two models on the same inputs."""
    results = []
    
    for sentence in test_sentences:
        # Get tokenized inputs
        inputs1 = tokenizer1(sentence, return_tensors="pt")
        inputs2 = tokenizer2(sentence, return_tensors="pt")
        
        # Run inference
        with torch.no_grad():
            outputs1 = model1(**inputs1)
            outputs2 = model2(**inputs2)
        
        # Compare hidden states (last layer)
        hidden1 = outputs1.last_hidden_state
        hidden2 = outputs2.last_hidden_state
        
        # Calculate differences
        if hidden1.shape == hidden2.shape:
            diff = torch.abs(hidden1 - hidden2)
            max_diff = torch.max(diff).item()
            mean_diff = torch.mean(diff).item()
            
            results.append({
                'sentence': sentence,
                'max_diff': max_diff,
                'mean_diff': mean_diff,
                'shapes_match': True
            })
        else:
            results.append({
                'sentence': sentence,
                'shapes_match': False,
                'shape1': hidden1.shape,
                'shape2': hidden2.shape
            })
    
    return results

def compare_tokenizers(tokenizer1, tokenizer2):
    """Compare two tokenizers."""
    # Check if tokenizers have the same vocabulary size
    vocab_size1 = tokenizer1.vocab_size
    vocab_size2 = tokenizer2.vocab_size
    
    if vocab_size1 != vocab_size2:
        print(f"Tokenizers have different vocabulary sizes: {vocab_size1} vs {vocab_size2}")
        return False
    
    # Compare vocabularies (some tokenizers may not expose vocab directly)
    try:
        vocab1 = tokenizer1.get_vocab()
        vocab2 = tokenizer2.get_vocab()
        
        if set(vocab1.keys()) != set(vocab2.keys()):
            print(f"Tokenizers have different vocabularies")
            return False
        
        # Check if token IDs match
        for token, id1 in vocab1.items():
            id2 = vocab2.get(token)
            if id1 != id2:
                print(f"Token '{token}' has different IDs: {id1} vs {id2}")
                return False
        
        return True
    except:
        print("Could not directly compare tokenizer vocabularies")
        
        # Alternative test: encode some sample text and compare results
        test_strings = [
            "Hello world",
            "This is a test of the tokenizers.",
            "Machine learning is fascinating!"
        ]
        
        for test_str in test_strings:
            ids1 = tokenizer1.encode(test_str)
            ids2 = tokenizer2.encode(test_str)
            
            if ids1 != ids2:
                print(f"Tokenizers produce different encodings for: '{test_str}'")
                print(f"Encoding 1: {ids1}")
                print(f"Encoding 2: {ids2}")
                return False
        
        print("Tokenizers produce identical encodings for test samples")
        return True

def compare_config(model1, model2):
    """Compare the configuration of two models."""
    try:
        config1 = model1.config.to_dict()
        config2 = model2.config.to_dict()
        
        # Find differences in configuration
        diffs = []
        all_keys = set(list(config1.keys()) + list(config2.keys()))
        
        for key in all_keys:
            val1 = config1.get(key, "NOT PRESENT")
            val2 = config2.get(key, "NOT PRESENT")
            
            if val1 != val2:
                diffs.append((key, val1, val2))
        
        if diffs:
            print("Configuration differences found:")
            for key, val1, val2 in diffs:
                print(f"  {key}: {val1} vs {val2}")
            return False
        else:
            print("Model configurations are identical")
            return True
    except:
        print("Could not compare model configurations")
        return None

def compare_huggingface_models(model_name1, model_name2):
    """
    Compare two Hugging Face models to determine if they are the same.
    
    Args:
        model_name1: Name or path of the first model
        model_name2: Name or path of the second model
        
    Returns:
        A dictionary with comparison results
    """
    print(f"Comparing models: {model_name1} vs {model_name2}")
    
    # Load models and tokenizers
    model1, tokenizer1 = load_model_and_tokenizer(model_name1)
    model2, tokenizer2 = load_model_and_tokenizer(model_name2)
    
    if model1 is None or model2 is None:
        return {"error": "Failed to load one or both models"}
    
    results = {
        "model_names": [model_name1, model_name2],
        "same_model": False,
        "details": {}
    }
    
    # Compare architectures
    print("\n=== Architecture Comparison ===")
    same_architecture = compare_model_architectures(model1, model2)
    results["details"]["same_architecture"] = same_architecture
    
    # Compare model configurations
    print("\n=== Configuration Comparison ===")
    same_config = compare_config(model1, model2)
    results["details"]["same_config"] = same_config
    
    # Compare tokenizers
    print("\n=== Tokenizer Comparison ===")
    same_tokenizer = compare_tokenizers(tokenizer1, tokenizer2)
    results["details"]["same_tokenizer"] = same_tokenizer
    
    # Compare weights (only if architecture is the same)
    if same_architecture:
        print("\n=== Weight Comparison ===")
        same_weights, weight_diffs = compare_model_weights(model1, model2)
        results["details"]["same_weights"] = same_weights
        results["details"]["weight_differences"] = [
            {"layer": wd[0], "max_diff": wd[1], "mean_diff": wd[2]} 
            for wd in weight_diffs[:10]  # Include only top 10 differences
        ]
    
    # Compare model outputs on sample inputs
    print("\n=== Output Comparison ===")
    test_sentences = [
        "Hello world!",
        "This is a test of the model comparison.",
        "Machine learning models can be compared in various ways."
    ]
    output_comparison = compare_model_outputs(model1, model2, tokenizer1, tokenizer2, test_sentences)
    results["details"]["output_comparison"] = output_comparison
    
    # Determine if models are the same
    if same_architecture and same_config and same_tokenizer:
        if 'same_weights' in results["details"] and results["details"]["same_weights"]:
            results["same_model"] = True
            print("\n✅ The models appear to be identical")
        else:
            print("\n⚠️ The models have the same architecture but different weights")
            # Check if the differences are very small (could be due to floating point precision)
            if 'weight_differences' in results["details"]:
                max_diff = max([wd["max_diff"] for wd in results["details"]["weight_differences"]])
                if max_diff < 1e-5:
                    print("The weight differences are very small (< 1e-5) and might be due to floating point precision")
                    results["same_model"] = "almost"
    else:
        print("\n❌ The models are different")
    
    return results

# Example usage:
model1_name = "nasa-impact/indus-sde-v0.1"
model2_name = "/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/general_analysis/artifacts/model-ytxjrbhy:v1"

results = compare_huggingface_models(model1_name, model2_name)
print(json.dumps(results, indent=2))

Comparing models: nasa-impact/indus-sde-v0.1 vs /rhome/sawale/indus_traning/mlm-fine-tuning/mlm/general_analysis/artifacts/model-ytxjrbhy:v1


Some weights of RobertaModel were not initialized from the model checkpoint at nasa-impact/indus-sde-v0.1 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at /rhome/sawale/indus_traning/mlm-fine-tuning/mlm/general_analysis/artifacts/model-ytxjrbhy:v1 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Architecture Comparison ===

=== Configuration Comparison ===
Configuration differences found:
  _name_or_path: nasa-impact/indus-sde-v0.1 vs /rhome/sawale/indus_traning/mlm-fine-tuning/mlm/general_analysis/artifacts/model-ytxjrbhy:v1

=== Tokenizer Comparison ===

=== Weight Comparison ===
Top 5 layer differences (max absolute difference):
pooler.dense.weight: max_diff=0.13650194, mean_diff=0.02256605
embeddings.word_embeddings.weight: max_diff=0.00000000, mean_diff=0.00000000
embeddings.position_embeddings.weight: max_diff=0.00000000, mean_diff=0.00000000
embeddings.token_type_embeddings.weight: max_diff=0.00000000, mean_diff=0.00000000
embeddings.LayerNorm.weight: max_diff=0.00000000, mean_diff=0.00000000

=== Output Comparison ===

❌ The models are different
{
  "model_names": [
    "nasa-impact/indus-sde-v0.1",
    "/rhome/sawale/indus_traning/mlm-fine-tuning/mlm/general_analysis/artifacts/model-ytxjrbhy:v1"
  ],
  "same_model": false,
  "details": {
    "same_architecture": 